# PostgreSQL (Enterprise AI System Design)

One of the most common mistakes in GenAI interviews is saying:

> **"We'll store everything in the vector database."**

A Senior AI Engineer knows that **every storage technology has a different responsibility**.

---

# 1. What is PostgreSQL?

## Definition

**PostgreSQL** is an **open-source relational database (RDBMS)** used to store structured and transactional data.

It stores data in **tables** with rows and columns and follows **ACID properties**.

---

## Interview Answer

> "PostgreSQL is a relational database used to store structured business data such as users, chat history, document metadata, audit logs, permissions, and application configuration. It provides transactional consistency and ACID compliance."

---

# Azure vs AWS

| Azure | AWS |
|--------|-----|
| Azure Database for PostgreSQL | Amazon RDS PostgreSQL |

---

# 2. Why PostgreSQL?

Suppose you're building an HR AI Assistant.

Should you store

- Users
- Roles
- Chat History
- Login
- Audit Logs

inside Qdrant?

**No.**

These belong in PostgreSQL.

---

# 3. Enterprise Architecture

```text id="9f4w0v"
                            User
                              │
                              ▼
      Azure Front Door / Route53 + CloudFront
                              │
                              ▼
Azure API Management / Amazon API Gateway
                              │
                              ▼
Azure App Gateway / AWS ALB
                              │
                              ▼
FastAPI (Container Apps) / ECS Fargate
                              │
            ┌─────────────────┼─────────────────┐
            ▼                 ▼                 ▼
        PostgreSQL         Redis           LangGraph
            │                                  │
            ▼                                  ▼
     User & Chat Data                   Retriever
                                               │
                                               ▼
                             Azure AI Search / Qdrant / OpenSearch
                                               │
                                               ▼
                              Azure OpenAI / AWS Bedrock
```

---

# 4. What Should Be Stored?

## Users

```text id="r1it1g"
User ID

Name

Email

Role

Department
```

---

## Chat History

```text id="kbry6g"
Question

Answer

Timestamp

User ID
```

---

## Document Metadata

```text id="gpb5ti"
Document Name

Owner

Upload Time

S3 Path

Status
```

---

## Feedback

```text id="c1v21v"
Question

Answer

Thumbs Up

Thumbs Down
```

---

## Audit Logs

```text id="n0zwhq"
Login

Logout

API Call

User Action
```

---

# 5. What Should NOT Be Stored?

❌ Embeddings

❌ PDF Files

❌ Images

❌ Audio

❌ Large Documents

Those belong in

- S3 / Blob Storage
- Qdrant
- OpenSearch
- Azure AI Search

---

# 6. Why Not Store PDFs?

Wrong

```text id="yqv8p5"
PostgreSQL

↓

100 MB PDF
```

Bad

Storage becomes expensive

Queries become slower

---

Correct

```text id="g48m4g"
Amazon S3

↓

PDF

↓

PostgreSQL

↓

Only Metadata
```

Example

```text id="v7m1o7"
File Name

S3 URL

Owner

Upload Date
```

---

# 7. Example Database Design

## User Table

```text id="jx7x7x"
user_id

name

email

role
```

---

## Chat Table

```text id="pwyh0r"
chat_id

user_id

question

answer

created_at
```

---

## Document Table

```text id="k10z5q"
doc_id

filename

s3_path

uploaded_by
```

---

# 8. SQL Example

Create Table

```sql id="n4yjku"
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    name TEXT,
    email TEXT
);
```

Insert

```sql id="22kjlwm"
INSERT INTO users(name,email)
VALUES
('Suraj','suraj@gmail.com');
```

---

# 9. Python Example

Install

```bash id="n9l82x"
pip install sqlalchemy psycopg2-binary
```

Model

```python id="yocv0q"
from sqlalchemy import Column, Integer, String
from database import Base

class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True)
    name = Column(String)
```

---

# Repository

```python id="5qgpxr"
def save_user(db, user):
    db.add(user)
    db.commit()
```

---

# 10. Chat History Flow

```text id="6v7ofj"
User

↓

FastAPI

↓

LangGraph

↓

Bedrock

↓

Generate Answer

↓

Save

↓

PostgreSQL
```

---

# 11. PostgreSQL vs Redis

| PostgreSQL | Redis |
|------------|--------|
| Permanent | Temporary |
| Disk | RAM |
| ACID | Cache |
| Structured | Key-Value |
| Business Data | Sessions |

---

# 12. PostgreSQL vs Qdrant

| PostgreSQL | Qdrant |
|------------|---------|
| Structured Data | Embeddings |
| SQL | Similarity Search |
| Chat History | Vector Search |
| Users | Semantic Search |

---

# 13. PostgreSQL vs S3

| PostgreSQL | S3 |
|------------|----|
| Metadata | Files |
| Tables | Objects |
| SQL Queries | Storage |

---

# 14. ACID

Interviewers often ask:

### What is ACID?

**A – Atomicity**

Transaction completes fully or rolls back.

**C – Consistency**

Database remains valid.

**I – Isolation**

Concurrent transactions don't interfere.

**D – Durability**

Committed data survives crashes.

---

# 15. Why ACID Matters?

Suppose

Payroll AI

updates salary.

Crash

during update.

Without ACID

Salary

↓

Corrupted

With ACID

Rollback.

---

# 16. Best Practices

✅ Store metadata only

✅ Store chat history

✅ Normalize tables

✅ Use indexes

✅ Connection Pooling

✅ Backups

---

# 17. Common Mistakes

❌ Store PDFs

❌ Store embeddings

❌ Store sessions

❌ Store cache

❌ Store images

---

# 18. Interview Questions

### Q1. Why PostgreSQL?

Store structured business data with transactional guarantees.

---

### Q2. Why not Redis?

Redis is temporary.

PostgreSQL is permanent.

---

### Q3. Why not Vector DB?

Vector DB stores embeddings.

Not business records.

---

### Q4. Why not S3?

S3 stores files.

Not relational data.

---

### Q5. What do you store?

Users

Chat History

Roles

Permissions

Metadata

Audit Logs

---

### Q6. How do PostgreSQL and Qdrant work together?

```text id="2y6w3j"
PDF

↓

S3

↓

Chunk

↓

Embedding

↓

Qdrant

↓

Question

↓

Retrieve Chunks

↓

Bedrock

↓

Answer

↓

Save Chat

↓

PostgreSQL
```

Qdrant stores **embeddings**, while PostgreSQL stores the **conversation record and business metadata**.

---

# 19. Scenario-Based Question

### Interviewer

> User uploads a PDF. Where do you store everything?

### Expected Answer

| Component | Storage |
|-----------|----------|
| PDF | Amazon S3 / Azure Blob |
| Metadata | PostgreSQL |
| Embeddings | Qdrant / OpenSearch / Azure AI Search |
| Chat History | PostgreSQL |
| Cache | Redis |

---

# 20. EPAM Senior Answer (2 Minutes)

> "In enterprise AI applications, PostgreSQL is the system of record for structured business data. I use it to store users, roles, chat history, document metadata, audit logs, feedback, and application configuration. Large files such as PDFs are stored in Amazon S3 or Azure Blob Storage, while their metadata is maintained in PostgreSQL. Vector embeddings are indexed in Qdrant, Amazon OpenSearch, or Azure AI Search for semantic retrieval. Redis is used for caching and session management. This separation of concerns ensures that each technology is used for the workload it is optimized for, resulting in a scalable, maintainable, and production-ready architecture."